# Stage 2 — Fine-tune on YOUR data (lower LR)

Loads the stage 1 checkpoint from **HuggingFace Hub**, continues training on your
labeled data at 1/10 the learning rate.

Two phases:
- Phase A (2 epochs, linear probe @ 1e-4): head only
- Phase B (5 epochs, full fine-tune @ 1e-6): everything

**Input data format**: folder-per-class
```
data/user/
  male/*.jpg
  female/*.jpg
```


## 0. Setup

In [ ]:
!pip install -q opencv-python-headless ultralytics transformers h5py scipy \
                pyyaml matplotlib huggingface_hub

import torch, os, json, time, math, csv, shutil, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Using device:', DEVICE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_CKPT_DIR = '/content/drive/MyDrive/IPD_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Drive mounted.')


## 1. Config

In [ ]:
CONFIG = {
    'seed': 42,
    'stage2': {
        'hf_repo': 'abhshkp/footfall-analysis-vit-stage1',  # ← EDIT: stage 1 source
        'output_dir': '/content/checkpoints/stage2',
        'user_data_root': '/content/data/user',
        'val_ratio': 0.15,
        'phase_a_epochs': 2, 'phase_a_head_lr': 1e-4,
        'phase_b_epochs': 5, 'phase_b_head_lr': 1e-6, 'phase_b_backbone_lr': 1e-6,
        'batch_size': 32, 'eval_batch_size': 64, 'num_workers': 4,
        'weight_decay': 0.3, 'warmup_ratio': 0.2, 'grad_clip': 1.0,
        'fp16': True, 'log_every': 20,
    },
    'hf_repo_stage2': 'abhshkp/footfall-gender-vit-retail-finetuned',  # ← EDIT: stage 2 target
}
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
print('Stage 1 source (HF):', CONFIG['stage2']['hf_repo'])
print('Stage 2 target (HF):', CONFIG['hf_repo_stage2'])


## 2. Helper functions

In [ ]:
from transformers import ViTImageProcessor
_VIT_PROC = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

def _to_tensor(pil_img):
    out = _VIT_PROC(images=pil_img, return_tensors='pt')
    return out['pixel_values'][0]

def train_transform(pil_img):
    if torch.rand(1).item() < 0.5:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    from PIL import ImageEnhance
    pil_img = ImageEnhance.Brightness(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    pil_img = ImageEnhance.Contrast(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    pil_img = ImageEnhance.Color(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    return _to_tensor(pil_img)

def val_transform(pil_img):
    return _to_tensor(pil_img)

print('Transforms ready.')

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset

GENDER_TO_IDX = {'female': 0, 'male': 1}
IDX_TO_GENDER = {0: 'female', 1: 'male'}

class CSVDataset(Dataset):
    def __init__(self, csv_path, image_root, transform=None):
        self.image_root = Path(image_root); self.transform = transform; self.rows = []
        with open(csv_path) as fh:
            for r in csv.DictReader(fh):
                g = r['gender'].strip().lower()
                if g not in GENDER_TO_IDX: continue
                p = Path(r['image'])
                if not p.is_absolute(): p = self.image_root / p
                if not p.exists(): continue
                self.rows.append((p, GENDER_TO_IDX[g]))
        if not self.rows: raise RuntimeError(f'No valid rows in {csv_path}')
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        p, lab = self.rows[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None: img = self.transform(img)
        return img, lab

class FolderPerClassDataset(Dataset):
    def __init__(self, root, transform=None, exts=('.jpg', '.jpeg', '.png')):
        self.root = Path(root); self.transform = transform; self.rows = []
        for name, idx in GENDER_TO_IDX.items():
            d = self.root / name
            if not d.exists(): continue
            for ext in exts:
                for img in d.glob(f'*{ext}'): self.rows.append((img, idx))
                for img in d.glob(f'*{ext.upper()}'): self.rows.append((img, idx))
        if not self.rows: raise RuntimeError(f'No images in {self.root}/{{male,female}}/')
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        p, lab = self.rows[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None: img = self.transform(img)
        return img, lab

class ConcatDataset(Dataset):
    def __init__(self, datasets):
        self.datasets = datasets; self.cum = [0]
        for d in datasets: self.cum.append(self.cum[-1] + len(d))
    def __len__(self): return self.cum[-1]
    def __getitem__(self, idx):
        for i, c in enumerate(self.cum[1:]):
            if idx < c: return self.datasets[i][idx - self.cum[i]]
        raise IndexError(idx)

def stratified_split(rows, val_ratio=0.1, seed=42):
    g = torch.Generator().manual_seed(seed)
    by_class = {}
    for i, (_, lab) in enumerate(rows): by_class.setdefault(lab, []).append(i)
    train_idx, val_idx = [], []
    for lab, idxs in by_class.items():
        perm = torch.randperm(len(idxs), generator=g).tolist()
        n_val = int(len(idxs) * val_ratio)
        for j, p in enumerate(perm):
            (val_idx if j < n_val else train_idx).append(idxs[p])
    return train_idx, val_idx

print('Datasets ready.')

In [ ]:
import torch.nn as nn
from transformers import ViTModel

class GenderClassifier(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224', num_labels=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained(vit_model_name)
        hidden_dim = self.vit.config.hidden_size  # 768
        self.head = nn.Linear(hidden_dim, num_labels)
        self.num_labels = num_labels
        self.id2label = {0: 'female', 1: 'male'}
        self.label2id = {'female': 0, 'male': 1}
        self.vit_model_name = vit_model_name

    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        return self.head(out.pooler_output)

    def freeze_backbone(self):
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.head.parameters(): p.requires_grad = True

    def unfreeze_backbone(self):
        for p in self.parameters(): p.requires_grad = True

    def param_groups(self, head_lr, backbone_lr):
        head_p, backbone_p = [], []
        for name, p in self.named_parameters():
            if not p.requires_grad: continue
            (head_p if name.startswith('head.') else backbone_p).append(p)
        groups = []
        if head_p: groups.append({'params': head_p, 'lr': head_lr, 'name': 'head'})
        if backbone_p: groups.append({'params': backbone_p, 'lr': backbone_lr, 'name': 'backbone'})
        return groups

    def save(self, out_dir):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), out_dir / 'pytorch_model.bin')
        cfg = {'architectures': ['GenderClassifier'], 'model_type': 'vit_gender',
               'vit_model_name': self.vit_model_name, 'num_labels': self.num_labels,
               'id2label': self.id2label, 'label2id': self.label2id}
        with open(out_dir / 'config.json', 'w') as f: json.dump(cfg, f, indent=2)
        print(f'Saved checkpoint to {out_dir}')

    @classmethod
    def load(cls, ckpt_dir, device='cpu'):
        with open(Path(ckpt_dir) / 'config.json') as f: cfg = json.load(f)
        m = cls(cfg['vit_model_name'], cfg.get('num_labels', 2))
        state = torch.load(Path(ckpt_dir) / 'pytorch_model.bin', map_location=device)
        m.load_state_dict(state, strict=False)
        m.to(device).eval()
        return m

print('Model class ready.')

In [ ]:
def _cosine_with_warmup(opt, warmup, total):
    from torch.optim.lr_scheduler import LambdaLR
    def lam(step):
        if step < warmup: return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * prog)))
    return LambdaLR(opt, lam)

def evaluate(model, loader, device):
    model.eval(); nc, nt, mc, mt, fc, ft = 0, 0, 0, 0, 0, 0
    with torch.no_grad():
        for imgs, labs in loader:
            imgs, labs = imgs.to(device), labs.to(device)
            with torch.amp.autocast('cuda', enabled=(device=='cuda')):
                logits = model(imgs)
            preds = logits.argmax(-1)
            nc += int((preds==labs).sum()); nt += labs.size(0)
            for l, p in zip(labs.tolist(), preds.tolist()):
                if l == 1: mt += 1; mc += int(p==1)
                else: ft += 1; fc += int(p==0)
    return {'accuracy': nc/max(nt,1), 'male_acc': mc/max(mt,1),
            'female_acc': fc/max(ft,1), 'n_total': nt}

def train_phase(model, train_loader, val_loader, device, phase_name,
                epochs, head_lr, backbone_lr, freeze,
                weight_decay=0.01, warmup_ratio=0.1, grad_clip=1.0,
                fp16=True, log_every=50, output_dir='.', best_acc=0.0):
    if freeze:
        model.freeze_backbone(); print(f'[{phase_name}] backbone FROZEN, head only')
    else:
        model.unfreeze_backbone(); print(f'[{phase_name}] backbone UNFROZEN, full fine-tune')
    opt = torch.optim.AdamW(model.param_groups(head_lr, backbone_lr), weight_decay=weight_decay)
    total_steps = epochs * len(train_loader)
    warmup = int(warmup_ratio * total_steps)
    sched = _cosine_with_warmup(opt, warmup, total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(fp16 and device=='cuda'))
    step, t0, history = 0, time.time(), []
    for ep in range(epochs):
        model.train(); rloss, rcorr, rtot = 0.0, 0, 0
        for imgs, labs in train_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(fp16 and device=='cuda')):
                logits = model(imgs)
                loss = nn.functional.cross_entropy(logits, labs)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            rloss += loss.item()*imgs.size(0)
            rcorr += int((logits.argmax(-1)==labs).sum()); rtot += imgs.size(0)
            if step % log_every == 0:
                lr = opt.param_groups[0]['lr']
                eta = (time.time()-t0)/step*(total_steps-step)
                print(f'  [{phase_name}] ep{ep+1}/{epochs} step {step}/{total_steps} '
                      f'loss={rloss/rtot:.4f} acc={rcorr/rtot:.3f} lr={lr:.2e} eta={eta:.0f}s')
        m = evaluate(model, val_loader, device)
        print(f'  [{phase_name}] ep{ep+1} VAL acc={m["accuracy"]:.4f} '
              f'male={m["male_acc"]:.3f} female={m["female_acc"]:.3f} n={m["n_total"]}')
        history.append({'phase': phase_name, 'epoch': ep+1, **m})
        if m['accuracy'] > best_acc:
            best_acc = m['accuracy']
            print(f'  [{phase_name}] NEW BEST, saving to {output_dir}/best')
            model.save(os.path.join(output_dir, 'best'))
    return best_acc, history

print('Trainer ready.')

## 3. Pull stage 1 checkpoint from HuggingFace Hub

In [ ]:
from huggingface_hub import snapshot_download

stage1_dir = '/content/checkpoints/stage1/best'
print(f'Downloading stage 1 from HF Hub: {CONFIG["stage2"]["hf_repo"]}')
snapshot_download(
    repo_id=CONFIG['stage2']['hf_repo'],
    local_dir=stage1_dir,
    repo_type='model'
)
print(f'Downloaded to: {stage1_dir}')
!ls -lh {stage1_dir}

with open(f'{stage1_dir}/config.json') as f:
    cfg = json.load(f)
print(f'\nBackbone: {cfg["vit_model_name"]}')
print(f'Labels:   {cfg["id2label"]}')


## 4. Upload Roboflow export + crop to folder-per-class
    Expects Roboflow YOLOv8 export zip with:
    data.yaml (has 'names:' list — your class names)
    train/images/*.jpg + train/labels/*.txt (YOLO format: class x y w h, normalized)
    valid/images/*.jpg + valid/labels/*.txt
 
    The cell reads data.yaml to map class IDs → gender names automatically.
    Just make sure your Roboflow classes are 'male' and 'female' (in any order).


In [ ]:
from google.colab import files
import zipfile, yaml
from pathlib import Path
import cv2

print('Upload your Roboflow export zip (YOLOv8 format)')
up = files.upload()
rf_zip = list(up.keys())[0]
print(f'Extracting {rf_zip}...')
with zipfile.ZipFile(rf_zip) as zf:
    zf.extractall('/content/data/roboflow')

# Read data.yaml to get class names
rf_root = Path('/content/data/roboflow')
yaml_files = list(rf_root.rglob('data.yaml'))
if not yaml_files:
    raise FileNotFoundError('No data.yaml found in Roboflow export')
with open(yaml_files[0]) as f:
    rf_cfg = yaml.safe_load(f)
class_names = rf_cfg.get('names', [])
print(f'Roboflow classes: {class_names}')

# Map class ID → gender name (handles both list and dict formats)
if isinstance(class_names, list):
    id_to_gender = {i: str(n).lower() for i, n in enumerate(class_names)}
elif isinstance(class_names, dict):
    id_to_gender = {int(k): str(v).lower() for k, v in class_names.items()}
else:
    raise ValueError(f'Unexpected data.yaml names format: {type(class_names)}')

# Verify classes are male/female
valid_genders = {'male', 'female'}
found_genders = set(id_to_gender.values())
if not found_genders.issubset(valid_genders):
    print(f'WARNING: classes include non-gender labels: {found_genders - valid_genders}')
    print(f'Will only crop boxes labeled male/female. Others skipped.')

# Prepare output folders
out_dir = Path('/content/data/user')
if out_dir.exists():
    import shutil
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'male').mkdir(exist_ok=True)
(out_dir / 'female').mkdir(exist_ok=True)

# Find all image/label folder pairs
image_dirs = sorted([p for p in rf_root.rglob('images') if p.is_dir()])
label_dirs = sorted([p for p in rf_root.rglob('labels') if p.is_dir()])
print(f'Found {len(image_dirs)} image folder(s), {len(label_dirs)} label folder(s)')

n_cropped = 0
n_skipped = 0
for img_dir, lbl_dir in zip(image_dirs, label_dirs):
    print(f'\nProcessing: {img_dir.relative_to(rf_root)}')
    for img_path in sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.JPG')) + sorted(img_dir.glob('*.png')):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            n_skipped += 1
            continue
        frame = cv2.imread(str(img_path))
        if frame is None:
            print(f'  Could not read {img_path.name}, skipping')
            n_skipped += 1
            continue
        h, w = frame.shape[:2]
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(float(parts[0]))
                x, y, bw, bh = map(float, parts[1:])
                gender = id_to_gender.get(cls_id)
                if gender not in ('male', 'female'):
                    continue
                # YOLO format: center x, y, width, height (normalized)
                x1 = int((x - bw/2) * w)
                y1 = int((y - bh/2) * h)
                x2 = int((x + bw/2) * w)
                y2 = int((y + bh/2) * h)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if x2 - x1 < 10 or y2 - y1 < 10:
                    continue  # too small, skip
                crop = frame[y1:y2, x1:x2]
                out_name = f'{img_path.stem}_{n_cropped:05d}.jpg'
                cv2.imwrite(str(out_dir / gender / out_name), crop)
                n_cropped += 1
    print(f'  running total: {n_cropped} crops')

# Summary
print('\n=== DONE ===')
for g in ('male', 'female'):
    n = len(list((out_dir / g).glob('*.jpg')))
    print(f'  {g}: {n} crops')
print(f'  total: {n_cropped} crops, skipped {n_skipped} images')
print(f'\nData ready at: {out_dir}')
print(f'Now run the next cell (Section 5: Train Stage 2) as normal.')

## 5. Train Stage 2

In [ ]:
s2 = CONFIG['stage2']

full_val = FolderPerClassDataset(s2['user_data_root'], transform=val_transform)
full_train = FolderPerClassDataset(s2['user_data_root'], transform=train_transform)
tr_idx, va_idx = stratified_split(full_val.rows, s2['val_ratio'], CONFIG['seed'])
print(f'Stage 2: train={len(tr_idx)} val={len(va_idx)}')
if len(tr_idx) < 50:
    print('WARNING: very small dataset. Consider lowering LR further.')

s2_train_loader = DataLoader(Subset(full_train, tr_idx),
    batch_size=s2['batch_size'], shuffle=True,
    num_workers=s2['num_workers'], pin_memory=True, drop_last=True)
s2_val_loader = DataLoader(Subset(full_val, va_idx),
    batch_size=s2['eval_batch_size'], shuffle=False,
    num_workers=s2['num_workers'], pin_memory=True)
print('Dataloaders ready.')


In [ ]:
print(f'Loading stage 1 checkpoint from {stage1_dir}')
model = GenderClassifier.load(stage1_dir, device=DEVICE)
os.makedirs(s2['output_dir'], exist_ok=True)

best_acc_s2 = 0.0
history_s2 = []

# Phase A: linear probe @ 1/10 stage 1 LR
best_acc_s2, h = train_phase(model, s2_train_loader, s2_val_loader, DEVICE,
    phase_name='S2-A', epochs=s2['phase_a_epochs'],
    head_lr=s2['phase_a_head_lr'], backbone_lr=0.0, freeze=True,
    weight_decay=s2['weight_decay'], warmup_ratio=s2['warmup_ratio'],
    grad_clip=s2['grad_clip'], fp16=s2['fp16'], log_every=s2['log_every'],
    output_dir=s2['output_dir'], best_acc=best_acc_s2)
history_s2.extend(h)

# Phase B: full fine-tune @ 1/10 stage 1 LR
best_acc_s2, h = train_phase(model, s2_train_loader, s2_val_loader, DEVICE,
    phase_name='S2-B', epochs=s2['phase_b_epochs'],
    head_lr=s2['phase_b_head_lr'], backbone_lr=s2['phase_b_backbone_lr'],
    freeze=False,
    weight_decay=s2['weight_decay'], warmup_ratio=s2['warmup_ratio'],
    grad_clip=s2['grad_clip'], fp16=s2['fp16'], log_every=s2['log_every'],
    output_dir=s2['output_dir'], best_acc=best_acc_s2)
history_s2.extend(h)

model.save(os.path.join(s2['output_dir'], 'final'))
with open(os.path.join(s2['output_dir'], 'history.json'), 'w') as f:
    json.dump(history_s2, f, indent=2)
print(f'\n=== STAGE 2 COMPLETE ===')
print(f'Best val accuracy: {best_acc_s2:.4f}')


In [ ]:
import matplotlib.pyplot as plt
if history_s2:
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    x = range(len(history_s2))
    ax.plot(x, [h['accuracy'] for h in history_s2], 'o-', label='overall')
    ax.plot(x, [h['male_acc'] for h in history_s2], 's--', label='male')
    ax.plot(x, [h['female_acc'] for h in history_s2], '^--', label='female')
    ax.set_xlabel('epoch'); ax.set_ylabel('accuracy')
    ax.set_title('Stage 2 val accuracy'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()


## 6. Save stage 2 to Google Drive + HuggingFace Hub

In [ ]:
stage2_drive = os.path.join(DRIVE_CKPT_DIR, 'stage2')
if os.path.exists(stage2_drive): shutil.rmtree(stage2_drive)
shutil.copytree(os.path.join(s2['output_dir'], 'best'), stage2_drive)
print(f'Copied to Drive: {stage2_drive}')


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
import os as _os
if 'HF_TOKEN' in _os.environ:
    api.login(_os.environ['HF_TOKEN'])
else:
    api.login()
try:
    api.create_repo(repo_id=CONFIG['hf_repo_stage2'], repo_type='model', private=False)
    print(f'Created repo: {CONFIG["hf_repo_stage2"]}')
except Exception as e:
    print(f'Repo may already exist: {e}')
api.upload_folder(
    folder_path=os.path.join(s2['output_dir'], 'best'),
    repo_id=CONFIG['hf_repo_stage2'],
    repo_type='model',
    commit_message='Stage 2 ViT checkpoint (fine-tuned on user data)'
)
print(f'\nUploaded to: https://huggingface.co/{CONFIG["hf_repo_stage2"]}')


## Done

Stage 2 checkpoint is on:
- Google Drive: `/content/drive/MyDrive/IPD_checkpoints/stage2/`
- HuggingFace Hub: `https://huggingface.co/abhshkp/footfall-gender-vit-retail-finetuned`

**Next**: open `test_model.ipynb` to test on image/video.
